## 1. Data loading & Initial inspection

In [1]:
import pandas as pd

import numpy as np

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

df.head(30)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   object 
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.shape

(7043, 21)

## 2. Data profiling & Quality inspection: Inspect the structure, missing values, hidden whitespace, and distinct categorical values.


In [5]:

data_quality_report = pd.DataFrame({
    'Data_Type': df.dtypes,
    'Null_Count': df.isnull().sum(),
    'Blank_Space_Count': (df == ' ').sum(),
    'Unique_Values': df.nunique()
})

data_quality_report

,Data_Type,Null_Count,Blank_Space_Count,Unique_Values
customerID,object,0,0,7043
gender,object,0,0,2
SeniorCitizen,int64,0,0,2
Partner,object,0,0,2
Dependents,object,0,0,2
tenure,int64,0,0,73
PhoneService,object,0,0,2
MultipleLines,object,0,0,3
InternetService,object,0,0,3
OnlineSecurity,object,0,0,3


In [6]:
# Inspect value counts for all categorical features
for col in df.select_dtypes(include=['object']).columns:
    print(f"=== Column: {col} (Unique: {df[col].nunique()}) ===")
    print(df[col].value_counts(dropna=False))
    print("-" * 50)

=== Column: customerID (Unique: 7043) ===
customerID
7590-VHVEG    1
3791-LGQCY    1
6008-NAIXK    1
5956-YHHRX    1
5365-LLFYV    1
             ..
9796-MVYXX    1
2637-FKFSY    1
1552-AAGRX    1
4304-TSPVK    1
3186-AJIEK    1
Name: count, Length: 7043, dtype: int64
--------------------------------------------------
=== Column: gender (Unique: 2) ===
gender
Male      3555
Female    3488
Name: count, dtype: int64
--------------------------------------------------
=== Column: Partner (Unique: 2) ===
Partner
No     3641
Yes    3402
Name: count, dtype: int64
--------------------------------------------------
=== Column: Dependents (Unique: 2) ===
Dependents
No     4933
Yes    2110
Name: count, dtype: int64
--------------------------------------------------
=== Column: PhoneService (Unique: 2) ===
PhoneService
Yes    6361
No      682
Name: count, dtype: int64
--------------------------------------------------
=== Column: MultipleLines (Unique: 3) ===
MultipleLines
No                  3390

In [7]:
# Summary statistics for numerical columns
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


### Key profiling findings:
1. **SeniorCitizen**: Stored as numeric `0`/`1`; needs mapping to `'No'`/`'Yes'` for readability.
2. **TotalCharges**: Stored as `object` data type instead of numeric due to 11 blank spaces (`' '`)
3. **Categorical Features**: Service columns contain clean and consistent categories (`Yes`, `No`, `No internet service` / `No phone service`).

## 3. Data cleaning

In [8]:
#SeniorCitizen

df['SeniorCitizen'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'})


In [11]:
#TotalCharges: 11 blank spaces 

blank_rows = df[df['TotalCharges'] == ' '][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]
blank_rows

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


In [12]:
blank_rows.shape

(11, 4)

In [13]:
len(df[df['tenure'] == 0])


11

In [14]:

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')

df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])



In [15]:
# Verify data type and check for remaining missing values
print("TotalCharges Data Type:", df['TotalCharges'].dtype)
print("Missing values in TotalCharges:", df['TotalCharges'].isnull().sum())

TotalCharges Data Type: float64
Missing values in TotalCharges: 0


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 4. Feature engineering
In this section, I create new features from existing columns to support cohort analysis and visual slicing in Power BI.

In [18]:
# Create tenure groups
labels = ['< 1 Year', '1-2 Years', '2-3 Years', '3-4 Years', '4-5 Years', '5-6 Years']
df['Tenure_Group'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 36, 48, 60, 72], labels=labels)

# Inspect the newly created column
df[['tenure', 'Tenure_Group']].head(10)

,tenure,Tenure_Group
0,1,< 1 Year
1,34,2-3 Years
2,2,< 1 Year
3,45,3-4 Years
4,2,< 1 Year
5,8,< 1 Year
6,22,1-2 Years
7,10,< 1 Year
8,28,2-3 Years
9,62,5-6 Years


## 5. Dataset export

In [ ]:
df.to_csv('telco_customer_churn_cleaned.csv', index=False)